### 1. Figure out a unifying and practical method for recording your ideas, confusions, questions, and plans.

To record my ideas and plans, I will use a Google Doc or Word document alongside a Jupyter Notebook. The document will serve as a space for notes, planning, and conceptual thinking, while the Jupyter Notebook will contain my code and experimental work. Any questions or points of confusion that arise during development will be documented in the same notes document, along with their corresponding explanations or resolutions. For each major project update, I will maintain a README.md file within the repository to summarize progress, objectives, and key decisions. Minor project updates will be tracked using a combination of GitHub issues, descriptive commit messages, and in-line code comments to ensure clear documentation and traceability throughout development.

### 2. Create abstract classes and/or use inheritance for a finite Markov (plain/reward) process. Consider your data structure choices.

In [5]:
"""
Below is an example of using the FiniteMRP class from src/finite_mrp.py. It
demonstrates how to create a Finite Markov Reward Process (MRP) and use it to
compute the value function using the value iteration algorithm.
"""

from pprint import pprint
from src.finite_mrp import FiniteMRP

# S is the finite set of states
S = {"A", "B", "C"}

# P[s][s'] is the transition probability 
P = {
    "A": {"A": 0.6, "B": 0.3, "C": 0.1},
    "B": {"A": 0.3, "B": 0.6, "C": 0.1},
    "C": {"C": 1.0}
}

# r[s] is the expected reward
r = {
    "A": 1.0,
    "B": 2.0,
    "C": -1.0
}

gamma = 0.9

mrp = FiniteMRP(S, P, r, gamma)

print("States:")
pprint(mrp.states())

for s in mrp.states():
    for s_next in mrp.states():
        prob = mrp.transition_prob(s, s_next)
        if prob > 0:
            print(f"P({s_next} | {s}) = {prob}")

for s in mrp.states():
    print(f"Reward r({s}) = {mrp.reward(s)}")

print("Discount factor gamma:")
pprint(mrp.discount())


States:
{'B', 'C', 'A'}
P(B | B) = 0.6
P(C | B) = 0.1
P(A | B) = 0.3
P(C | C) = 1.0
P(B | A) = 0.3
P(C | A) = 0.1
P(A | A) = 0.6
Reward r(B) = 2.0
Reward r(C) = -1.0
Reward r(A) = 1.0
Discount factor gamma:
0.9


### 3. Have a method for value convergence, like we did in class, via matrix form or element-wise where s is a state and s′ is a possible next-step state. You can use eigenvalues but be ready to deal with degenerate cases (for example, ([0 1], [1 0])). When coming up with a method (or several methods) for value convergence, consider how your method scales with the number of states

In [6]:
"""
Below is an example of using the value_iteration_vectorized function from the 
value_convergence module found in src/value_convergence.py. It runs the value 
iteration algorithm and returns the converged value function and the number of 
iterations it took to converge. It runs for a maximum of 10000 iterations and 
a tolerance of 1e-6 from the mrp defined above in question 2.
"""

from src.value_convergence import value_iteration_vectorized

V, iters = value_iteration_vectorized(mrp)

print("Converged value function:")
for s, v in V.items():
    print(f"V({s}) = {v:.4f}")

print(f"Converged in {iters} iterations")

Converged value function:
V(B) = 3.8428
V(C) = -10.0000
V(A) = 2.4730
Converged in 133 iterations


### 4. Create an initialization for your classes corresponding to basic grid worlds.

In [ ]:
"""
Below is an example of how to intialize a grid world from
any given matrix. The meat of the workings are in
grid_world.py
"""

from src.grid_world import gridworld_from_mask
from src.value_convergence import value_iteration_vectorized
from src.finite_mrp import FiniteMRP

import numpy as np

mask = np.array([
    [0,    -np.inf, -1, -1     , -1],
    [-1,   -np.inf, -1, -np.inf, -1],
    [-1,   -np.inf, -1, -np.inf, -1],
    [-1,   -1     , -1, -np.inf, -1],
    [-3,   -3     , -3, -3     , -3],
])

# Convert mask to S, P, r
# if num_actions == 4:
#     directions = N, E, S, W
# if num_actions == 8:
#     directions = N, NE, E, SE, S, SW, W, NW
S, P, r = gridworld_from_mask(mask, num_actions=8)

# Create FiniteMRP
mrp = FiniteMRP(S, P, r, gamma=0.9)
# Run value iteration convergence
V, iters = value_iteration_vectorized(mrp)


### 5. Is γ really necessary here? Do we need the same γ for every time step, t? Why are we multiplying by γ and not γ^2? Think it through. Experiment with different values of γ. Does it make an impact on what values converge to per state?

Unless non-convergence is acceptable, a discount factor γ is necessary. This is supported by testing, as γ approaches 1.0, convergence no longer occurs, and the iterative Bellman update implemented runs indefinitely. In this setup, the same γ is applied at every update step, with its exponent increasing over time, so future rewards are progressively discounted. If γ were initialized at γ^2, those future rewards would be discounted twice as heavily. Changes in γ directly affect the number of iterations required for convergence, because γ determines how far into the future the process effectively values rewards. As γ increases, the process places greater weight on distant future rewards, which slows convergence and can prevent it entirely when γ is too close to 1.

### 6. So far, there is no control over transitions and no hope that will change, regardless of how the rewards are initially given. How would you go about adding actions into the mix? Think about it.

In the case of 2D grid worlds, the available action space would be some combination of transitions between differing grid spaces. For simplicity, I have implemented the actions of N, E, S, W corresponding to the cardinal directions. To control which actions get taken, you can take advantage of states and value convergence. These two things combined with some form of path planning policy allow for the agent to begin taking educated movements.

In [ ]:
"""
Below is an example of running a maze solver with a actions.
The available actions are defined in grid_world.py. The grid
world script has been updated from question 4 to now also
include the ability to acount for actions. There is
also a greedy policy attatched to determine best action.
"""


from src.grid_world import gridworld_mdp_from_mask
from src.value_convergence import value_iteration_mdp
from src.finite_mdp import FiniteMDP
from src.policy import greedy_policy
from src.animate import animate_policy

import numpy as np

from IPython.display import HTML

mask = np.array([
    [1,    -np.inf, -1, -1     , -1],
    [-1,   -np.inf, -1, -np.inf, -1],
    [-1,   -np.inf, -1, -np.inf, -1],
    [-1,   -1     , -1, -np.inf, -1],
    [-3,   -3     , -3, -3     , -3],
    # [-2,   -2     , -2, -2     , -2],
])

S, A, P, r, terminal_states = gridworld_mdp_from_mask(mask)

print(terminal_states)

mdp = FiniteMDP(S, A, P, r, gamma=0.2)

V, _ = value_iteration_mdp(mdp)
policy = greedy_policy(mdp, V)

ani = animate_policy((4, 4), mdp, policy, mask, terminal_states)
HTML(ani.to_jshtml())


{(0, 0)}


### 7. (optional challenge - team + LLM friendly) Try to determine the right implementation or a complex like what is shown below without resorting to 3D initializations with large memory footprints. See if you can further generalize your design via a cartesian product of matrix tiles along a binary tree of height 3. There is currently a real gap in the number of available gridworld options that corresponding to “Lego”-like tilings, such as what is found with trees of grids and simplicial complexes more generaly. Such innovations could really boost the availability of options corresponding to reinforcement learning coupled with procedurally generated environments. Moreover, pivotal-irreversible decision making is tied to the structure of trees.

I have a couple of ideas for how something like this might work but essentially they boil down to the creation of some kind of mapping function that references only available transtion locations from one level of depth to the next. This would allow for a smaller amount of memory usage while maintaining the ability for exploration.

### 8. Give some thoughts about the whole Markov process model. Do you like it? Do you think there is a better or different way to model random decision making? Considerwhat you think are the advantages and limitations to markov processes. Reinforcement learning tends to focus on markov processes, but that does not mean there cannot bemore innovative ideas.

The markov process seems to be a mathematically sound framework for early machine learning. However, it also seems to have its share of limitations, these would include things like struggling with partial observations. The markov process also seems like it would struggle with evolving states and a changing environment. This makes the process good for something like stationary path planning and bad for something like navigating real life or procedurally generated situations. 